# Additional experiments

Run in order. Each stage checkpoints and tees its log to `results/<stem>.log`. Datasets and provenance: `PROVENANCE.md`. Results write-up: `REPORT.md`.

In [ ]:
%matplotlib inline

import importlib.metadata as meta
import os, platform, sys

print(f"python        {sys.version.split()[0]}")
print(f"platform      {platform.platform()}")
print(f"cpu count     {os.cpu_count()}")
for name in ("numpy", "scipy", "scikit-learn", "pandas", "lightgbm",
             "featuretools", "openfe", "autofeat", "pmlb", "rdata", "beamfeat"):
    try:
        print(f"{name:<13} {meta.version(name)}")
    except meta.PackageNotFoundError:
        print(f"{name:<13} MISSING")
print(f"interpreter   {sys.executable}")

In [ ]:
# datasets from canonical sources (network access required)
!python fetch_data.py --all

## Datasets

In [ ]:
import pathlib
import numpy as np, pandas as pd

DATA = pathlib.Path("data")
SPECS = [
    ("communities",   "communities_crime_numeric.csv", "last",         "violent-crime rate"),
    ("superconduct",  "superconductivity.csv",         "critical_temp","critical temperature (K)"),
    ("tecator",       "tecator.csv",                   "fat",          "fat content (%)"),
    ("eyedata",       "eyedata.csv",                   "trim32",       "TRIM32 expression"),
    ("riboflavin",    "riboflavin.csv",                "first",        "log production rate"),
    ("ct_slices",     "ct_slices.csv",                 "last",         "axial slice position"),
    ("blogfeedback",  "blogfeedback.csv",              "last",         "comment count"),
    ("ujiindoorloc",  "ujiindoorloc.csv",              "longitude",    "longitude (m)"),
    ("geomusic",      "geomusic.csv",                  "last",         "latitude of origin"),
]

def load(path, target):
    df = pd.read_csv(path)
    if target == "last":  y = df.iloc[:, -1]; X = df.iloc[:, :-1]
    elif target == "first": y = df.iloc[:, 0]; X = df.iloc[:, 1:]
    else: y = df[target]; X = df.drop(columns=target)
    if path.name == "tecator.csv":
        X = X[[c for c in X.columns if c.startswith(("_", "absorbance"))]]
    return X, y

rows, shapes = [], {}
for name, fname, target, desc in SPECS:
    p = DATA / fname
    if not p.exists():
        rows.append((name, "-", "-", desc, "absent (fetch_data.py --all)")); continue
    X, y = load(p, target)
    shapes[name] = (len(X), X.shape[1])
    rows.append((name, len(X), X.shape[1], desc,
                 f"target {y.min():.3g} to {y.max():.3g}"))
overview = pd.DataFrame(rows, columns=["dataset", "n", "p", "target", "range / status"])
overview

In [ ]:
def probe(X, y, max_rows=3000, max_cols=300, seed=0):
    """Sub-sample rows and columns for the pairwise statistics."""
    rng = np.random.default_rng(seed)
    ri = rng.choice(len(X), min(len(X), max_rows), replace=False)
    ci = rng.choice(X.shape[1], min(X.shape[1], max_cols), replace=False)
    A = X.to_numpy(float)[np.ix_(ri, ci)]
    return A[:, A.std(axis=0) > 0], y.to_numpy(float)[ri]

rows, cache = [], {}
for name, fname, target, desc in SPECS:
    path = DATA / fname
    if not path.exists():
        rows.append(dict(dataset=name, status="absent")); continue
    X, y = load(path, target)
    A, yy = probe(X, y)
    with np.errstate(all="ignore"):
        C = np.corrcoef(A, rowvar=False)
        off = np.abs(C[np.triu_indices_from(C, k=1)]); off = off[np.isfinite(off)]
        Az = (A - A.mean(0)) / A.std(0)
        yz = (yy - yy.mean()) / (yy.std() or 1.0)
        marginal = np.abs(Az.T @ yz) / len(yy)
    cache[name] = (A, yy, off, marginal)
    rows.append(dict(dataset=name, n=len(X), p=X.shape[1],
                     missing=int(X.isna().sum().sum()),
                     constant=int((X.nunique() <= 1).sum()),
                     dup_rows=int(X.duplicated().sum()),
                     target_skew=round(float(pd.Series(y).skew()), 2),
                     med_absr=round(float(np.median(off)), 3),
                     p95_absr=round(float(np.quantile(off, 0.95)), 3),
                     max_absr=round(float(off.max()), 4),
                     med_marginal=round(float(np.median(marginal)), 3)))
diagnostics = pd.DataFrame(rows)
diagnostics

In [ ]:
import matplotlib.pyplot as plt
from make_figures import (use_acs_style, ACCENT, ACCENT_LT, WARN, NEUTRAL,
                          GRID, COL_DOUBLE)

use_acs_style()
fig, axes = plt.subplots(1, 2, figsize=(COL_DOUBLE, 2.4))

ax = axes[0]                                   # where each dataset sits in (n, p)
for name, (n, p) in shapes.items():
    ax.scatter(n, p, s=24, color=ACCENT, zorder=3)
    ax.annotate(name, (n, p), textcoords="offset points", xytext=(5, 3), fontsize=6)
lim = [40, 80_000]
ax.plot(lim, lim, linestyle="--", linewidth=0.8, color=NEUTRAL)
ax.annotate("p = n", (lim[0] * 1.3, lim[0] * 1.6), fontsize=6.5, color=NEUTRAL)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(lim); ax.set_ylim(20, 8000)
ax.set_xlabel("rows n"); ax.set_ylabel("input columns p")
ax.grid(True, axis="both", color=GRID, linewidth=0.5)

ax = axes[1]                                   # tecator: adjacent absorbance channels
tp = DATA / "tecator.csv"
if tp.exists():
    X, _ = load(tp, "fat")
    A = X.to_numpy(float)
    for row in A[::12]:
        ax.plot(range(1, 101), row, linewidth=0.5, color=ACCENT_LT, alpha=0.45)
    r = np.corrcoef(A[:, 49], A[:, 50])[0, 1]
    ax.annotate(f"adjacent channels: r = {r:.5f}", (0.04, 0.9),
                xycoords="axes fraction", fontsize=6.5, color="#333333")
    ax.set_xlabel("absorbance channel"); ax.set_ylabel("absorbance")
    ax.grid(True, axis="y", color=GRID, linewidth=0.5)
else:
    ax.axis("off")
pathlib.Path("figures").mkdir(exist_ok=True)
fig.tight_layout()
fig.savefig("figures/datasets_overview.pdf"); fig.savefig("figures/datasets_overview.png", dpi=300)
plt.show()

In [ ]:
present = list(cache)
SERIES = ["#14496E", "#B4552B", "#4E86B4", "#5F6E79", "#8D99A3",
          "#2E7D5B", "#7A5C99", "#A8873B", "#C0504D"]

fig, axes = plt.subplots(1, 3, figsize=(COL_DOUBLE, 2.2))

ax = axes[0]                                    # dependence among candidates
for i, name in enumerate(present):
    xs = np.sort(cache[name][2]); ys = np.arange(1, len(xs) + 1) / len(xs)
    step = max(1, len(xs) // 1500)
    ax.plot(xs[::step], ys[::step], linewidth=1.1, color=SERIES[i % len(SERIES)], label=name)
ax.set_xlabel("|r| between feature pairs"); ax.set_ylabel("cumulative fraction")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.grid(True, axis="both", color=GRID, linewidth=0.5)

ax = axes[1]                                    # a correlation matrix with visible structure
pick = "communities" if "communities" in cache else present[0]
A = cache[pick][0][:, :70]
order = np.argsort(np.corrcoef(A, rowvar=False)[0])
im = ax.imshow(np.corrcoef(A[:, order], rowvar=False), cmap="RdBu_r",
               vmin=-1, vmax=1, interpolation="nearest")
ax.set_title(f"{pick}: 70 features, ordered", fontsize=7)
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

ax = axes[2]                                    # marginal association with the target
box_kw = dict(widths=0.6, patch_artist=True,
              flierprops=dict(marker=".", markersize=1.2, alpha=0.4))
try:                                            # matplotlib >= 3.10
    bp = ax.boxplot([cache[n][3] for n in present], orientation="horizontal", **box_kw)
except TypeError:                               # older releases
    bp = ax.boxplot([cache[n][3] for n in present], vert=False, **box_kw)
ax.set_yticks(range(1, len(present) + 1), present)
for patch in bp["boxes"]:
    patch.set_facecolor(ACCENT_LT); patch.set_edgecolor(ACCENT); patch.set_linewidth(0.7)
for med in bp["medians"]: med.set_color(WARN); med.set_linewidth(1.2)
for w in bp["whiskers"] + bp["caps"]: w.set_color(NEUTRAL); w.set_linewidth(0.7)
# |r| is n-dependent, so mark what independence alone would produce:
# E|r| = sqrt(2 / (pi (n - 1))) for each dataset's probe sample
null_level = [np.sqrt(2 / (np.pi * (cache[n][0].shape[0] - 1))) for n in present]
ax.scatter(null_level, range(1, len(present) + 1), marker="|", s=90,
           color=NEUTRAL, zorder=4, label="expected under independence")
ax.legend(fontsize=5.5, loc="upper center", bbox_to_anchor=(0.5, -0.28),
          frameon=False)
ax.set_xlabel("|corr(feature, target)|"); ax.tick_params(axis="y", labelsize=6)
ax.grid(True, axis="x", color=GRID, linewidth=0.5)

handles, labels = axes[0].get_legend_handles_labels()   # one legend under all panels
fig.legend(handles, labels, loc="lower center", ncols=min(5, len(labels)),
           fontsize=6, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.10, 1, 1))
fig.savefig("figures/diagnostics_dependence.pdf"); fig.savefig("figures/diagnostics_dependence.png", dpi=300)
plt.show()

## Comparison on high-dimensional data

The baselines and beamfeat over nine datasets, p = 81 to 4,088, at a 900 s per-fit budget.

In [ ]:
!python bench.py highdim ridge_raw,rf_raw,lgbm_raw,beamfeat,beamfeat_ridge 5 results/highdim_fast.json 2>&1 | tee results/highdim_fast.log | grep --line-buffered -E 'R2=|ERROR|Traceback|wrote'

In [ ]:
# Label-permuted controls at p = 4,088: the selector must return nothing.
from beamfeat import BeamFeatTransformer

path = pathlib.Path("data/riboflavin.csv")
if path.exists():
    df = pd.read_csv(path)
    X, y = df.iloc[:, 1:].to_numpy(float), df.iloc[:, 0].to_numpy(float)
    rng = np.random.default_rng(0)
    counts = []
    for s in range(5):
        m = BeamFeatTransformer(random_state=s).fit(X, rng.permutation(y))
        counts.append(len(m.formulas()))
        print(f"permuted control {s}: selected {counts[-1]}")
    print(f"controls selecting nothing: {sum(c == 0 for c in counts)}/5")
else:
    print("riboflavin.csv absent; run fetch_data.py first")

## What the certified features carry

The comparison hands every constructor's output to the same linear model, so
that differences reflect the features rather than the consumer. That leaves
one question unasked: whether the certified features hold structure a tree
ensemble cannot extract for itself, and whether a shortfall against the tree
baselines is a property of the features or of the consumer they are given to.

The grid below is fixed, not searched: the same three consumers on every
dataset, on the raw columns and on the certified features. The highest cell
in a row is not a result: choosing a consumer by test score is selection on
the test set, and the reported method keeps ridge, whose coefficients are the
deliverable. What the grid is for is the direction of the gap. Where the
certified features close most of the distance to the tree baselines, the
shortfall was the consumer; where they do not, it was the screening, and the
signal is too diffuse for a sparsity-oriented guarantee to keep.

The rows, caps and splits come from the comparison harness itself, so the
raw columns here reproduce its baseline numbers.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from beamfeat import BeamFeatTransformer

from bench import get_datasets      # identical rows, caps and seeds as the comparison

# every dataset in the comparison, cheapest first so an interrupted run
# still leaves a usable table
CASES = ["tecator_p100", "communities_p100", "superconduct_p81", "geomusic_p117",
         "eyedata_p200", "riboflavin_p4088", "blogfeedback_p280", "ct_slices_p384",
         "ujiindoorloc_p520"]
SPLITS = 3
POOL = get_datasets("highdim")

def consumers(A, ytr, B, yte):
    return {
        "ridge": make_pipeline(StandardScaler(),
                               RidgeCV(alphas=np.logspace(-4, 4, 25))).fit(A, ytr).score(B, yte),
        "rf": RandomForestRegressor(n_estimators=300, random_state=0,
                                    n_jobs=-1).fit(A, ytr).score(B, yte),
        "lgbm": lgb.LGBMRegressor(n_estimators=300, random_state=0,
                                  verbose=-1).fit(A, ytr).score(B, yte),
    }

rows = []
for name in CASES:
    if name not in POOL:
        print(f"{name:18s} skipped, not fetched", flush=True)
        continue
    Xd, yd, _ = POOL[name]
    for split in range(SPLITS):
        Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.25, random_state=split)
        raw = consumers(Xtr, ytr, Xte, yte)
        tf = BeamFeatTransformer(random_state=0).fit(Xtr, ytr)
        con = consumers(tf.transform(Xtr), ytr, tf.transform(Xte), yte)
        rows.append(dict(dataset=name, split=split, k=len(tf.formulas()),
                         **{f"raw_{a}": b for a, b in raw.items()},
                         **{f"bf_{a}": b for a, b in con.items()}))
        print(f"{name:18s} s{split}  k={len(tf.formulas()):3d}  "
              f"raw {raw['ridge']:.3f}/{raw['rf']:.3f}/{raw['lgbm']:.3f}  "
              f"bf {con['ridge']:.3f}/{con['rf']:.3f}/{con['lgbm']:.3f}", flush=True)

    pd.DataFrame(rows).to_json("results/consumer_ablation_rows.json")   # checkpoint per dataset

consumer_table = (pd.DataFrame(rows).drop(columns="split")
                  .groupby("dataset").mean().round(3).reindex(
                      [n for n in CASES if n in set(r["dataset"] for r in rows)]))
consumer_table["tree_gap_raw"] = (consumer_table[["raw_rf", "raw_lgbm"]].max(axis=1)
                                  - consumer_table["raw_ridge"]).round(3)
consumer_table["gap_closed"] = (consumer_table[["bf_rf", "bf_lgbm"]].max(axis=1)
                                - consumer_table["bf_ridge"]).round(3)
consumer_table.to_json("results/consumer_ablation.json")
consumer_table

## Search depth and robustness

Recovery of planted targets across depth, scorer and beam width, including targets whose intermediates carry zero population correlation with the target.

In [ ]:
!python depth_ladder.py --seeds 10 --out results/depth_ladder.json 2>&1 | tee results/depth_ladder.log

## Selection procedures

BH, BY, fixed-X and model-X knockoffs across four dependence regimes, reporting realised FDR under both null definitions.

In [ ]:
!python selector_comparison.py --trials 100 --out results/selector_comparison.json 2>&1 | tee results/selector_comparison.log
!python selector_comparison.py --trials 100 --m 100 --k 10 --out results/selector_comparison_m100.json 2>&1 | tee results/selector_comparison_m100.log

## Split stability

Thirty independent search/selection splits per dataset, selections compared up to value-equivalence, plus the multi-split aggregate.

In [ ]:
!python split_stability.py --splits 30 --out results/split_stability.json 2>&1 | tee results/split_stability.log

In [ ]:
!python multisplit.py data/tecator.csv fat --splits 20 2>&1 | tee results/multisplit_demo.log

## Scalability

Planted signal among p columns; time, peak memory, recovery and the FDR flag per cell, each in an isolated subprocess.

In [ ]:
!python scalability.py --p-grid 10,30,100,300,1000 --seeds 5 --out results/scalability.json 2>&1 | tee results/scalability.log

## Cost profile

In [ ]:
import cProfile, pstats, time

src = DATA / "superconductivity.csv"
if src.exists():
    frame = pd.read_csv(src).sample(5000, random_state=0)
    Xall = frame.drop(columns="critical_temp").to_numpy(float)
    yall = frame["critical_temp"].to_numpy(float)
    label = "superconductivity, 5,000 rows"
else:
    frame = pd.read_csv(DATA / "communities_crime_numeric.csv")
    Xall, yall = frame.iloc[:, :-1].to_numpy(float), frame.iloc[:, -1].to_numpy(float)
    label = "communities"

from beamfeat import BeamFeatRegressor

Xtr, Xte, ytr, yte = train_test_split(Xall, yall, test_size=0.25, random_state=0)
profiler = cProfile.Profile(); profiler.enable()
BeamFeatRegressor(random_state=0).fit(Xtr, ytr)
profiler.disable()

BUCKETS = [("standardisation", ("_standardise",)),
           ("summary statistics", ("_var", "_mean", "_count_reduce", "reduce", "fromnumeric")),
           ("expression evaluation", ("expression.py",)),
           ("matrix assembly", ("column_stack", "numpy.array", "_shape_base", "concatenate")),
           ("linear algebra", ("linalg", "dot", "matmul", "ridge")),
           ("redundancy pruning", ("_prune_redundant",)),
           ("permutation testing", ("selection.py",))]

stats = pstats.Stats(profiler)
seen, total = {}, 0.0
for (fn, line, name), (cc, nc, tt, ct, callers) in stats.stats.items():
    total += tt
    seen[f"{pathlib.Path(fn).name}:{name}"] = seen.get(f"{pathlib.Path(fn).name}:{name}", 0.0) + tt
rows = [(bucket, sum(v for k, v in seen.items() if any(key in k for key in keys)))
        for bucket, keys in BUCKETS]
rows.append(("everything else", max(total - sum(t for _, t in rows), 0.0)))
profile_table = (pd.DataFrame(rows, columns=["cost centre", "seconds"])
                 .assign(share=lambda d: (d.seconds / total * 100).round(1))
                 .sort_values("seconds", ascending=False).reset_index(drop=True))
print(f"{total:.1f}s profiled on {label}")
profile_table

In [ ]:
sweep = []
for hold in (0.5, 0.65, 0.8):
    for seed in range(3):
        Xa, Xb, ya, yb = train_test_split(Xall, yall, test_size=0.25, random_state=seed)
        t0 = time.time()
        model = BeamFeatRegressor(random_state=0, selection_holdout=hold).fit(Xa, ya)
        sweep.append(dict(holdout=hold, seed=seed, seconds=time.time() - t0,
                          r2=model.score(Xb, yb), features=len(model.formulas()),
                          search_rows=int(len(Xa) * (1 - hold))))
sweep = pd.DataFrame(sweep)
holdout_summary = sweep.groupby("holdout").agg(
    search_rows=("search_rows", "first"), seconds=("seconds", "mean"),
    r2_mean=("r2", "mean"), r2_sd=("r2", "std"), features=("features", "mean")).round(3)
holdout_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(COL_DOUBLE, 2.3),
                         gridspec_kw={"width_ratios": [1.5, 1, 1]})

ax = axes[0]                                   # where a fit spends its seconds
d = profile_table.sort_values("seconds")
vectorisable = ("standardisation", "summary statistics", "matrix assembly")
colours = [WARN if c in vectorisable else NEUTRAL if c == "everything else" else ACCENT
           for c in d["cost centre"]]
ax.barh(d["cost centre"], d["seconds"], color=colours, edgecolor="white", linewidth=0.5)
for y, (v, s) in enumerate(zip(d["seconds"], d["share"])):
    ax.annotate(f" {s:.0f}%", (v, y), va="center", fontsize=6, color=NEUTRAL)
ax.set_xlabel("seconds in one fit"); ax.tick_params(axis="y", labelsize=6)
ax.set_xlim(0, d["seconds"].max() * 1.25)
ax.grid(True, axis="x", color=GRID, linewidth=0.5); ax.grid(False, axis="y")

ax = axes[1]                                   # cost against the size of the search half
ax.plot(holdout_summary.index, holdout_summary["seconds"], marker="o", color=ACCENT)
ax.set_xlabel("selection holdout"); ax.set_ylabel("fit seconds")
ax.set_ylim(0, holdout_summary["seconds"].max() * 1.15)

ax = axes[2]                                   # and what it costs in accuracy
ax.errorbar(holdout_summary.index, holdout_summary["r2_mean"],
            yerr=holdout_summary["r2_sd"], marker="s", color=WARN,
            capsize=2, linewidth=1.2, elinewidth=0.8)
ax.set_xlabel("selection holdout"); ax.set_ylabel("held-out $R^2$")
for ax in axes[1:]:
    ax.grid(True, axis="y", color=GRID, linewidth=0.5)

fig.tight_layout()
fig.savefig("figures/cost_profile.pdf"); fig.savefig("figures/cost_profile.png", dpi=300)
plt.show()

## Tables and figures

In [ ]:
import glob, json, collections

rows = []
for f in glob.glob("results/highdim_*.json"):
    rows += json.load(open(f))
df = pd.DataFrame(rows)
if "error" not in df:
    df["error"] = None
ok = df[df["error"].isna()]
summary = (ok.groupby("method")
             .agg(mean_r2=("r2", "mean"), worst=("r2", "min"),
                  neg=("r2", lambda s: int((s < 0).sum())),
                  mean_features=("n_new", "mean"), mean_s=("seconds", "mean"))
             .sort_values("mean_r2", ascending=False).round(3))
errors = (df[df["error"].notna()].groupby("method")
            .error.apply(lambda s: collections.Counter(e.split(":")[0] for e in s)))
print(summary, "\n\nrecorded failures and budget hits:\n", errors, sep="")

piv = ok.pivot_table(index=["dataset", "split"], columns="method", values="r2")
print("\nper-dataset means:\n", piv.groupby("dataset").mean().round(3))

try:
    st = json.load(open("results/split_stability.json"))
    for k, v in st.items():
        print(f"{k:14s} R2 {v['r2_mean']:.3f}\u00b1{v['r2_std']:.3f} "
              f"Jaccard(eq) {v['jaccard_mean']:.2f} stable classes {len(v['stable_features'])}")
except FileNotFoundError:
    pass

In [ ]:
!python make_figures.py